# 🏆 [Day 36] 실전 정보 추출(IE) 및 개체명 인식(NER) 핸즈온 워크북

> **핵심 학습 목표**:
> 비정형 의학 텍스트에서 지식그래프 노드로 쓰일 **5대 핵심 개체(`Compound`, `Gene`, `Disease`, `Symptom`, `PharmacologicClass`)**를 정확히 포착하고, 규칙 기반의 한계를 극복하는 **LLM Pydantic 구조화 추출 및 3단계 정제 퍼널**을 직접 구축·검증합니다.
>
> 1. 🧐 **[개체 표현]**: 구간(Span)과 BIO 태깅의 메커니즘 및 `B-` 태그가 가르는 경계 보존 실측
> 2. 📐 **[규칙 기반 NER]**: 표준 사전(`name2id.json`)과 정규식 패턴 매칭, 대소문자/불용어 방어
> 3. 🚫 **[4대 한계점 검증]**: 미등록어(`statins`), 복합어 경계(`Warfarin-related`), 대소문자 오탐(`large`)
> 4. 🧠 **[LLM 구조화 추출]**: `with_structured_output` 및 Pydantic `EntityList` 기반 강제 타이핑
> 5. 🧪 **[3단계 정제 퍼널]**: 유형 검증(Type) ➔ 원문 등장 대조(Presence) ➔ 복합키 중복 제거(Dedup)
> 6. 📊 **[하이브리드 벤치마크]**: 규칙 기반 vs LLM 추출 결과 비교 및 지식그래프 연계 전략 도출

## 0. 환경 설정 및 API 키 확인

In [1]:
import os
import sys
import json
import re
from pathlib import Path
from collections import Counter
from typing import List, Dict, Any, Optional, Tuple, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# 현재 작업 디렉토리의 .env 로딩
env_file = Path('.env')
if env_file.exists():
    load_dotenv(env_file, override=True)
else:
    load_dotenv('../.env', override=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
MODEL_NAME = 'gpt-5.6-luna'

if not OPENAI_API_KEY:
    raise RuntimeError('OPENAI_API_KEY가 필요합니다. .env 파일을 확인하세요.')

print('✅ [환경 설정 완료] OpenAI API Key 준비 완료')
print(f'✅ [모델]: {MODEL_NAME}')

✅ [환경 설정 완료] OpenAI API Key 준비 완료
✅ [모델]: gpt-5.6-luna


## 1. 구간(Span) 및 BIO 태깅 인코딩/디코딩

In [2]:
def find_spans(text: str, entities: List[Tuple[str, str]]) -> List[Tuple[int, int, str]]:
    spans = []
    for name, ent_type in entities:
        start = text.find(name)
        if start != -1:
            spans.append((start, start + len(name), ent_type))
    return spans

def spans_to_bio(tokens: List[str], token_spans: List[Tuple[int, int]], entity_spans: List[Tuple[int, int, str]]) -> List[str]:
    tags = ['O'] * len(tokens)
    for ent_start, ent_end, ent_type in entity_spans:
        first = True
        for idx, (t_start, t_end) in enumerate(token_spans):
            if t_start >= ent_start and t_end <= ent_end:
                tags[idx] = f'B-{ent_type}' if first else f'I-{ent_type}'
                first = False
    return tags

def decode_bio(tokens: List[str], tags: List[str]) -> List[Tuple[str, str]]:
    entities = []
    curr_tokens, curr_type = [], None
    for token, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            if curr_tokens and curr_type:
                entities.append((' '.join(curr_tokens), curr_type))
            curr_tokens = [token]
            curr_type = tag.split('-')[1]
        elif tag.startswith('I-'):
            if curr_type == tag.split('-')[1]:
                curr_tokens.append(token)
        else:
            if curr_tokens and curr_type:
                entities.append((' '.join(curr_tokens), curr_type))
                curr_tokens, curr_type = [], None
    if curr_tokens and curr_type:
        entities.append((' '.join(curr_tokens), curr_type))
    return entities

tokens = ['Patients', 'taking', 'Simvastatin', 'developed', 'severe', 'myopathy', '.']
token_spans = [(0, 8), (9, 15), (16, 27), (28, 37), (38, 44), (45, 53), (53, 54)]
entity_spans = [(16, 27, 'Compound'), (45, 53, 'Disease')]

bio_tags = spans_to_bio(tokens, token_spans, entity_spans)
restored = decode_bio(tokens, bio_tags)

print(f'• 토큰 목록: {tokens}')
print(f'• BIO 태그:  {bio_tags}')
print(f'• 디코딩 복원: {restored}')
assert len(restored) == 2
print('✅ [PASS] 경계 보존 인코딩/디코딩 검증 통과')

• 토큰 목록: ['Patients', 'taking', 'Simvastatin', 'developed', 'severe', 'myopathy', '.']
• BIO 태그:  ['O', 'O', 'B-Compound', 'O', 'O', 'B-Disease', 'O']
• 디코딩 복원: [('Simvastatin', 'Compound'), ('myopathy', 'Disease')]
✅ [PASS] 경계 보존 인코딩/디코딩 검증 통과


## 2. 규칙 기반 NER 및 표준 사전(`name2id.json`) 대조

In [3]:
data_path = Path('data/name2id.json')
with open(data_path, 'r', encoding='utf-8') as f:
    n2id = json.load(f)

names = n2id.get('names', {})
symbols = n2id.get('symbols', {})
brand_stops = set(n2id.get('brand_stopwords', []))

print(f'• 표준 사전 로드 완료: 명칭 {len(names):,}건, 유전자 기호 {len(symbols):,}건')

sample_doc = "Patients with rs9923231 variant taking Simvastatin or Warfarin showed altered CYP3A4 metabolism."

rule_results = []
for w in re.findall(r'\b\w+\b', sample_doc):
    if w.lower() in brand_stops:
        continue
    if w in names:
        rule_results.append({'name': w, 'type': names[w].split('::')[0]})
    elif w in symbols and w.isupper():
        rule_results.append({'name': w, 'type': 'Gene'})

for v in re.findall(r'\brs[0-9]+\b', sample_doc):
    rule_results.append({'name': v, 'type': 'Variant'})

print(f'• 규칙 기반 추출 결과 ({len(rule_results)}건):')
for r in rule_results:
    print(f"  - [{r['type']}] {r['name']}")

• 표준 사전 로드 완료: 명칭 6,695건, 유전자 기호 13,113건
• 규칙 기반 추출 결과 (5건):
  - [Compound] Simvastatin
  - [Gene] CYP3A4
  - [Compound] Warfarin
  - [Variant] rs9923231


## 3. Pydantic 구조화 스키마 및 LLM 추출

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

EntityType = Literal["Compound", "Gene", "Disease", "Symptom", "PharmacologicClass", "Other"]

class ExtractedEntity(BaseModel):
    name: str = Field(description="원문에 등장한 개체의 정확한 텍스트 (변형 금지)")
    type: EntityType = Field(description="개체 유형 (5종 또는 Other)")
    other_type: Optional[str] = Field(default=None, description="type이 Other인 경우 실제 유형")
    confidence: Optional[float] = Field(default=None, description="0.0~1.0 확신도 점수")

class ExtractedEntityList(BaseModel):
    entities: List[ExtractedEntity] = Field(default_factory=list, description="추출된 개체 목록")

llm = ChatOpenAI(model=MODEL_NAME)
structured_llm = llm.with_structured_output(ExtractedEntityList)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract biomedical entities strictly into allowed types: Compound, Gene, Disease, Symptom, PharmacologicClass, Other."),
    ("user", "Text: {text}")
])

chain = prompt | structured_llm
test_text = "Statins including Simvastatin are metabolized by CYP3A4, reducing risk of myopathy in Clinical trial."
res = chain.invoke({"text": test_text})

print(f'• LLM 구조화 추출 원시 결과 ({len(res.entities)}건):')
for e in res.entities:
    print(f"  - [{e.type}] {e.name} (conf: {e.confidence})")

• LLM 구조화 추출 원시 결과 (5건):
  - [Compound] Simvastatin (conf: 0.98)
  - [Gene] CYP3A4 (conf: 0.95)
  - [PharmacologicClass] statins (conf: 0.92)
  - [Disease] myopathy (conf: 0.91)
  - [Other] Clinical trial (conf: 0.99)


## 4. 3단계 후처리 정제 퍼널 (Post-Processing Funnel)

In [5]:
ALLOWED_TYPES = {"Compound", "Gene", "Disease", "Symptom", "PharmacologicClass"}

def clean_entities(entities: List[Dict[str, Any]], raw_text: str):
    stage1 = [e for e in entities if e.get('type') in ALLOWED_TYPES]
    stage2 = [e for e in stage1 if e.get('name') in raw_text]
    seen = set()
    stage3 = []
    for e in stage2:
        key = (e.get('name'), e.get('type'))
        if key not in seen:
            seen.add(key)
            stage3.append(e)
    stats = {
        'raw': len(entities),
        'after_type': len(stage1),
        'after_presence': len(stage2),
        'final': len(stage3),
        'dropped_type': len(entities) - len(stage1),
        'dropped_hallucination': len(stage1) - len(stage2),
        'dropped_duplicate': len(stage2) - len(stage3),
    }
    return stage3, stats

raw_ents = [e.model_dump() for e in res.entities]
clean_ents, stats = clean_entities(raw_ents, test_text)

print('• [퍼널 감축 통계]')
print(f"  - 원시 추출: {stats['raw']}개")
print(f"  - 1단계 (유형 검증): {stats['after_type']}개 (탈락: {stats['dropped_type']}개)")
print(f"  - 2단계 (원문 등장): {stats['after_presence']}개 (탈락: {stats['dropped_hallucination']}개)")
print(f"  - 3단계 (중복 제거): {stats['final']}개 (탈락: {stats['dropped_duplicate']}개)")
print(f"✨ 최종 확정된 신뢰 개체: {[e['name'] for e in clean_ents]}")

• [퍼널 감축 통계]
  - 원시 추출: 5개
  - 1단계 (유형 검증): 4개 (탈락: 1개)
  - 2단계 (원문 등장): 4개 (탈락: 0개)
  - 3단계 (중복 제거): 4개 (탈락: 0개)
✨ 최종 확정된 신뢰 개체: ['Simvastatin', 'CYP3A4', 'statins', 'myopathy']


## 5. 규칙 기반 vs LLM 하이브리드 대조 벤치마크

In [6]:
rule_set = {r['name'] for r in rule_results if r['name'] in test_text}
llm_set = {e['name'] for e in clean_ents}

common = rule_set & llm_set
llm_only = llm_set - rule_set
rule_only = rule_set - llm_set

print('=' * 80)
print('📊 [하이브리드 NER 대조 벤치마크]')
print(f'• 공통 포착 개체 ({len(common)}건): {list(common)}')
print(f'• LLM 단독 포착 (미등록 복수형/약효군) ({len(llm_only)}건): {list(llm_only)}')
print(f'• 규칙 단독 포착 ({len(rule_only)}건): {list(rule_only)}')
print('=' * 80)
print('💡 [인사이트]: 복수형 명칭(\x27statins\x27)이나 사전 미등재 표현은 LLM이 정밀하게 보완하며,')
print('   3단계 퍼널을 통해 \x27Other\x27 타입은 안전하게 걸러져 지식그래프의 오염을 방지합니다.')

📊 [하이브리드 NER 대조 벤치마크]
• 공통 포착 개체 (2건): ['Simvastatin', 'CYP3A4']
• LLM 단독 포착 (미등록 복수형/약효군) (2건): ['statins', 'myopathy']
• 규칙 단독 포착 (0건): []
💡 [인사이트]: 복수형 명칭('statins')이나 사전 미등재 표현은 LLM이 정밀하게 보완하며,
   3단계 퍼널을 통해 'Other' 타입은 안전하게 걸러져 지식그래프의 오염을 방지합니다.
